<a href="https://colab.research.google.com/github/rjshrd/myrepos/blob/master/TextSummaryPDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install pdfplumber transformers

In [33]:
import os
import re
from typing import List, Dict
import pdfplumber
from transformers import AutoTokenizer, pipeline

In [34]:
TOKENIZER = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [35]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text.append(page.extract_text() or "")
    except Exception as e:
        print(f"Error extracting text from {pdf_path}: {e}")
    return "\n".join(text)

In [36]:
def preprocess_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [37]:
def chunk_text(text: str, max_tokens: int = 512) -> List[str]:
  sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)
  chunks, current_chunk = [], []
  current_length = 0

  for sentence in sentences:
    token_count = len(TOKENIZER.tokenize(sentence))
    if current_length + token_count <= max_tokens:
      current_chunk.append(sentence)
      current_length += token_count
    else:
      chunks.append(" ".join(current_chunk))
      current_chunk = [sentence]
      current_length = token_count

  if current_chunk: # Add the last chunk
    chunks.append(" ".join(current_chunk))

  return chunks

In [38]:
def process_pdf_to_chunks(pdf_path: str, max_tokens: int = 512) -> Dict[str, List[str]]:
  raw_text = extract_text_from_pdf(pdf_path)
  clean_text = preprocess_text(raw_text)
  chunks = chunk_text(clean_text, max_tokens)

  return {"pdf_file": os.path.basename(pdf_path),
          "chunks": chunks}


In [39]:
def process_multiple_pdfs(pdf_folder: str, max_tokens: int = 512) -> List[Dict[str, List[str]]]:
  pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
  results = []

  for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    print(f"Processing {pdf_path}....")
    result = process_pdf_to_chunks(pdf_path, max_tokens)
    results.append(result)

  return results

In [40]:
def save_chunks_to_files(results: List[Dict[str, List[str]]], output_folder: str):

  os.makedirs(output_folder, exist_ok=True)
  for result in results:
    output_file = os.path.join(output_folder, f"{result['pdf_file']}_chunks.txt")
    with open(output_file, "w", encoding="utf-8") as f:
      for idx, chunk in enumerate(result["chunks"]):
        f.write(f"Chunk {idx + 1}: \n{chunk}\n\n")
    print(f"Saved chunks to {output_file}")

In [41]:
if __name__ == "__main__":
  pdf_folder = "/content/pdfs"
  output_folder = "/content/chunks"

  results = process_multiple_pdfs(pdf_folder, max_tokens=512)
  save_chunks_to_files(results, output_folder)

Processing /content/pdfs/sonslovelawr00lawr.pdf....
Processing /content/pdfs/howtostudyincoll00pauk_1.pdf....


Token indices sequence length is longer than the specified maximum sequence length for this model (703 > 512). Running this sequence through the model will result in indexing errors


Saved chunks to /content/chunks/sonslovelawr00lawr.pdf_chunks.txt
Saved chunks to /content/chunks/howtostudyincoll00pauk_1.pdf_chunks.txt


In [23]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use cuda:0


In [24]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
              page_text = page.extract_text()
              if page_text:
                text.append(page_text)
    except Exception as e:
        print(f"Error extracting text from {pdf_path}: {e}")
    return "\n".join(text)

In [25]:
def preprocess_text(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text.strip())
    paragraphs = [para.strip() for para in text.split("\n") if para.strip()]
    return paragraphs

In [26]:
def chunk_text_semantically(paragraphs: List[str], max_tokens: int = 512) -> List[str]:
    chunks = []
    current_chunk = []
    current_length = 0

    for paragraph in paragraphs:
        token_count = len(paragraph.split())
        if current_length + token_count <= max_tokens:
            current_chunk.append(paragraph)
            current_length += token_count
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [paragraph]
            current_length = token_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [27]:
def summarize_chunks(chunks: List[str], min_length: int=50, max_length: int = 200) -> List[str]:
    summaries = []
    for i, chunk in enumerate(chunks):
      try:
        summary = summarizer(chunk, min_length=min_length, max_length=max_length, truncation=True)
        summaries.append(summary[0]['summary_text'])
      except Exception as e:
        print(f"Error summarizing chunk {i}: {e}")
        summaries.append("Error in summarizing chunk")
    return summaries

In [28]:
def process_pdf_to_summary(pdf_path: str, max_tokens: int = 512, min_length: int = 50, max_length: int = 200) -> Dict[str, List[str]]:
  raw_text = extract_text_from_pdf(pdf_path)
  paragraphs = preprocess_text(raw_text)
  chunks = chunk_text_semantically(paragraphs, max_tokens)
  summaries = summarize_chunks(chunks, min_length, max_length)

  return {"pdf_file": os.path.basename(pdf_path),
          "summaries": summaries}

In [29]:
def process_multiple_pdfs_to_summaries(pdf_folder: str, max_tokens: int = 512, min_length: int = 50, max_length: int = 200) -> List[Dict[str, List[str]]]:
  pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
  results = []

  for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    print(f"Processing {pdf_path}....")
    result = process_pdf_to_summary(pdf_path, max_tokens, min_length, max_length)
    results.append(result)

  return results

In [30]:
def save_summaries_to_files(results: List[Dict[str, List[str]]], output_folder: str):

  os.makedirs(output_folder, exist_ok=True)
  for result in results:
    output_file = os.path.join(output_folder, f"{result['pdf_file']}_summary.txt")
    with open(output_file, "w", encoding="utf-8") as f:
      for idx, summary in enumerate(result["summaries"]):
        f.write(f"Summary {idx + 1}: \n{summary}\n\n")

    print(f"Saved summaries to {output_file}")

In [31]:
if __name__ == "__main__":
  pdf_folder = "/content/pdfs"
  output_folder = "/content/summaries"

  results = process_multiple_pdfs_to_summaries(pdf_folder, max_tokens=512, min_length=50, max_length=200)
  save_summaries_to_files(results, output_folder)

Processing /content/pdfs/sonslovelawr00lawr.pdf....


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Your max_length is set to 200, but your input_length is only 3. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=1)


Error summarizing chunk 1: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Processing /content/pdfs/howtostudyincoll00pauk_1.pdf....
Error summarizing chunk 0: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error summarizing chunk 1: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Saved sum

In [1]:
!pip install pdfplumber pandas spacy transformers
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 50.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
import re
import json
import pandas as pd
from typing import List, Dict, Union
import pdfplumber
from transformers import pipeline
import spacy
from PIL import Image

In [13]:
NLP = spacy.load("en_core_web_sm")

In [3]:
SUMMARIZER = pipeline("summarization", model="facebook/bart-large-cnn")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [14]:
def extract_text_and_images_from_pdf(pdf_path: str, image_output_folder: str) -> (str, List[Image.Image], List[str]):
    text = []
    tables = []
    images = []
    image_descriptions = []
    os.makedirs(image_output_folder, exist_ok=True)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_number, page in enumerate(pdf.pages):
                # Extract text
                text.append(page.extract_text() or "")

                # Extract tables
                page_tables = page.extract_tables()
                for table in page_tables:
                    tables.append(pd.DataFrame(table))

                # Extract images
                for img in page.images:
                    img_bbox = (img['x0'], img['top'], img['x1'], img['bottom'])
                    cropped_img = page.within_bbox(img_bbox).to_image()
                    image_file = os.path.join(image_output_folder, f"{os.path.basename(pdf_path)}_page{page_number + 1}_img{len(images) + 1}.png")
                    cropped_img.save(image_file)
                    images.append(image_file)
                    image_descriptions.append(f"Image extracted from page {page_number + 1}, likely a graph or chart.")
    except Exception as e:
        print(f"Error extracting content from {pdf_path}: {e}")

    return "\n".join(text), tables, images, image_descriptions

In [15]:
def preprocess_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [16]:
def semantic_chunking(text: str, max_tokens: int = 512) -> List[str]:
    doc = NLP(text)
    paragraphs = [para.text.strip() for para in doc.sents]
    chunks, current_chunk = [], []
    current_length = 0

    for paragraph in paragraphs:
        token_count = len(paragraph.split())
        if current_length + token_count <= max_tokens:
            current_chunk.append(paragraph)
            current_length += token_count
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [paragraph]
            current_length = token_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [17]:
def summarize_chunks(chunks: List[str]) -> List[str]:
    summaries = []
    for chunk in chunks:
        try:
            summary = SUMMARIZER(chunk, max_length=130, min_length=30, do_sample=False)
            summaries.append(summary[0]["summary_text"])
        except Exception as e:
            print(f"Error summarizing chunk: {e}")
            summaries.append("Error summarizing this chunk.")
    return summaries

In [18]:
def summarize_tables(tables: List[pd.DataFrame]) -> List[str]:
    table_summaries = []
    for idx, table in enumerate(tables):
        try:
            table_summary = f"Table {idx + 1}: {len(table)} rows, {len(table.columns)} columns."
            table_summaries.append(table_summary)
        except Exception as e:
            table_summaries.append(f"Error summarizing table {idx + 1}: {e}")
    return table_summaries

In [19]:
def process_pdf_and_summarize(pdf_path: str, image_output_folder: str, max_tokens: int = 512) -> Dict[str, Union[str, List[str], List[Dict]]]:

    raw_text, tables, images, image_descriptions = extract_text_and_images_from_pdf(pdf_path, image_output_folder)
    clean_text = preprocess_text(raw_text)

    # Text processing
    text_chunks = semantic_chunking(clean_text, max_tokens)
    text_summaries = summarize_chunks(text_chunks)

    # Table processing
    table_summaries = summarize_tables(tables)

    return {
        "pdf_file": os.path.basename(pdf_path),
        "text_chunks": text_chunks,
        "text_summaries": text_summaries,
        "table_summaries": table_summaries,
        "images": images,
        "image_descriptions": image_descriptions
    }

In [20]:
def process_multiple_pdfs_and_summarize(pdf_folder: str, output_folder: str, max_tokens: int = 512) -> List[Dict[str, Union[str, List[str], List[Dict]]]]:

    pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
    results = []
    image_output_folder = os.path.join(output_folder, "images")
    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_folder, pdf_file)
        print(f"Processing and summarizing {pdf_path}...")
        result = process_pdf_and_summarize(pdf_path, image_output_folder, max_tokens)
        results.append(result)
    return results

In [21]:
def save_results(results: List[Dict[str, Union[str, List[str], List[Dict]]]], output_folder: str):

    os.makedirs(output_folder, exist_ok=True)
    for result in results:
        output_file = os.path.join(output_folder, f"{result['pdf_file']}_summary.json")
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=4)
        print(f"Saved results to {output_file}")

In [22]:
if __name__ == "__main__":
    pdf_folder = "/content/pdfs"
    output_folder = "/content/summaries_details"

    # Process PDFs and generate summaries
    results = process_multiple_pdfs_and_summarize(pdf_folder, output_folder, max_tokens=512)
    save_results(results, output_folder)

Processing and summarizing /content/pdfs/sonslovelawr00lawr.pdf...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processing and summarizing /content/pdfs/howtostudyincoll00pauk_1.pdf...


Your max_length is set to 130, but your input_length is only 7. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)


Error extracting content from /content/pdfs/howtostudyincoll00pauk_1.pdf: Bounding box (0.0, -5.175789965505828e-07, 418.55253, 639.175069482421) is not fully within parent page bounding box (0, 0.0, 418.55253, 639.1751)
Saved results to /content/summaries_details/sonslovelawr00lawr.pdf_summary.json
Saved results to /content/summaries_details/howtostudyincoll00pauk_1.pdf_summary.json
